In [63]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# =================================================================
# PASO 1: Cargar los Datos Ya Preprocesados
# =================================================================
df = pd.read_csv('data/partidos_entrenamiento.csv',parse_dates=['date'])

# =================================================================
# PASO 2: Definir las Características (X) y el Objetivo (y)
# =================================================================
# Incluimos de forma estricta los nuevos predictores que calcula tu script
features = [
    'goles_totales_marcados_local_10', 'goles_totales_concedidos_local_10', 'forma_puntos_local_10',
    'goles_totales_marcados_visitante_10', 'goles_totales_concedidos_visitante_10', 'forma_puntos_visitante_10',
    'dif_goles_marcados_10', 'dif_goles_concedidos_10', 'dif_forma_puntos_10',
    'puntos_fifa_local', 'puntos_fifa_visitante', 'dif_puntos_fifa',
    'ranking_pos_local', 'ranking_pos_visitante',
    'neutral', 'importancia_torneo'
]

df = df[df['date']< '2026-05-01']

df_train = df[df['date']< '2023-06-01'].copy()
X_train = df_train[features]
y_train = df_train['resultado']
df_test = df[df['date']>= '2023-06-01'].copy()
X_test = df_test[features]
y_test = df_test['resultado']
y_test


modelo_gb = HistGradientBoostingClassifier(
    max_iter=200,   #reducimos arboles
    learning_rate=0.05, #pasos más cortos
    class_weight='balanced',
    max_depth= 4,       #limitamos la profundidad
    min_samples_leaf=50, # Exigimos que al menos 30 partidos cumplan una regla para darla por válida
    random_state=42
)

# =================================================================
# PASO 2: Entrenar el modelo (Aprender de los datos de entrenamiento)
# =================================================================
# Ajustamos el modelo usando las pistas (X_train) y las respuestas reales (y_train)
modelo_gb.fit(X_train, y_train)

# =================================================================
# PASO 3: Realizar las predicciones 
# =================================================================
# Le pedimos al modelo que intente adivinar los resultados del 20% de partidos reservados
predicciones = modelo_gb.predict(X_test)

# =================================================================
# PASO 4: Evaluar los resultados en consola
# =================================================================
# Calculamos el porcentaje total de aciertos
precision = accuracy_score(y_test, predicciones)

print(f"La precisión es: {precision * 100:.2f}%")
print("\nPor cada resultado (1=Local, 0=Empate, 2=Visitante):")
print(classification_report(y_test, predicciones))



La precisión es: 55.82%

Por cada resultado (1=Local, 0=Empate, 2=Visitante):
              precision    recall  f1-score   support

         0.0       0.29      0.21      0.24       748
         1.0       0.68      0.66      0.67      1511
         2.0       0.53      0.66      0.59       960

    accuracy                           0.56      3219
   macro avg       0.50      0.51      0.50      3219
weighted avg       0.55      0.56      0.55      3219



'goles_totales_marcados_local_10', 'goles_totales_concedidos_local_10', 'forma_puntos_local_10',
    'goles_totales_marcados_visitante_10', 'goles_totales_concedidos_visitante_10', 'forma_puntos_visitante_10',
    'dif_goles_marcados_10', 'dif_goles_concedidos_10', 'dif_forma_puntos_10',
    'puntos_fifa_local', 'puntos_fifa_visitante', 'dif_puntos_fifa',
    'ranking_pos_local', 'ranking_pos_visitante',
    'neutral', 'importancia_torneo'

In [ ]:
import pandas as pd
import numpy as np

# 1. Cargamos el archivo limpio que generó tu script de preprocesamiento
df_entrenamiento = pd.read_csv('data/partidos_entrenamiento.csv', parse_dates=['date'])

# 2. Creamos un diccionario vacío para guardar el estado de cada país
diccionario_maestro = {}

# 3. Agrupamos todos los equipos que existen en la base de datos (tanto locales como visitantes)
todas_las_selecciones = set(df_entrenamiento['home_team'].unique()).union(set(df_entrenamiento['away_team'].unique()))

for seleccion in todas_las_selecciones:
    # Buscamos los partidos donde participó esta selección
    partidos_equipo = df_entrenamiento[(df_entrenamiento['home_team'] == seleccion) | (df_entrenamiento['away_team'] == seleccion)]
    
    if not partidos_equipo.empty:
        # Tomamos el último partido de su historia (el más reciente en la línea de tiempo)
        ultimo_partido = partidos_equipo.sort_values('date').iloc[-1]
        
        # Extraemos sus métricas dependiendo de si en ese último partido jugó como local o visitante
        if ultimo_partido['home_team'] == seleccion:
            goles_totales_marcados_10 = ultimo_partido['goles_totales_marcados_local_10']
            goles_totales_concedidos_10 = ultimo_partido['goles_totales_concedidos_local_10']
            forma_puntos_10 = ultimo_partido['forma_puntos_local_10']
            puntos_fifa = ultimo_partido['puntos_fifa_local']
            ranking_pos = ultimo_partido['ranking_pos_local']
        else:
            goles_totales_marcados_10 = ultimo_partido['goles_totales_marcados_visitante_10']
            goles_totales_concedidos_10 = ultimo_partido['goles_totales_concedidos_visitante_10']
            forma_puntos_10 = ultimo_partido['forma_puntos_visitante_10']
            puntos_fifa = ultimo_partido['puntos_fifa_visitante']
            ranking_pos = ultimo_partido['ranking_pos_visitante']
            
        # Guardamos su "ficha técnica" en el diccionario maestro
        diccionario_maestro[seleccion] = {
            'goles_recientes': goles_totales_marcados_10,
            'goles_concedidos': goles_totales_concedidos_10,
            'forma_reciente': forma_puntos_10,
            'puntos_fifa': puntos_fifa,
            'ranking_fifa': ranking_pos
        }


¡Diccionario Maestro creado con éxito! Se ha guardado el estado actual de 321 selecciones mundiales.


In [65]:
def predecir_encuentro_mundial(local, visitante):
    # 1. Verificar si ambas selecciones existen en el diccionario maestro
    if local not in diccionario_maestro or visitante not in diccionario_maestro:
        return f"Error: Uno o ambos equipos ('{local}' / '{visitante}') no tienen suficiente historial en el CSV."
        
    # 2. Extraer los datos guardados de cada país de forma automatizada
    loc = diccionario_maestro[local]
    vis = diccionario_maestro[visitante]
    
    # 3. Construir la fila de 11 predictores calculando las diferencias (Locales menos Visitantes)
    datos_partido = {
        'importancia_torneo': 4.0, # Nivel de máxima importancia (Mundial)
        'neutral': 1,      # 1 porque es campo neutral en el Mundial
        ##local
        'goles_totales_marcados_local_10': loc['goles_recientes'],
        'goles_totales_concedidos_local_10': loc['goles_concedidos'],
        'forma_puntos_local_10' : loc['forma_reciente'],
        'puntos_fifa_local' : loc ['puntos_fifa'],
        'ranking_pos_local' : loc ['ranking_fifa'],
        ##visitante
        'goles_totales_marcados_visitante_10': vis['goles_recientes'],
        'goles_totales_concedidos_visitante_10': vis['goles_concedidos'],
        'forma_puntos_visitante_10' : vis['forma_reciente'],
        'puntos_fifa_visitante' : vis ['puntos_fifa'],
        'ranking_pos_visitante' : vis ['ranking_fifa'],
        # Calculamos los predictores diferenciales automáticos
        'dif_goles_marcados_10': loc['goles_recientes'] - vis['goles_recientes'],
        'dif_forma_puntos_10': loc['forma_reciente'] - vis['forma_reciente'],
        'dif_goles_concedidos_10': loc['goles_concedidos'] - vis['goles_concedidos'],
        'dif_puntos_fifa': loc['puntos_fifa'] - vis['puntos_fifa'],
        'dif_ranking_fifa': loc['ranking_fifa'] - vis['ranking_fifa']
    }
    
    # Convertimos en el DataFrame estructurado que exige tu HistGradientBoosting
    X_partido = pd.DataFrame([datos_partido])
    
    # 4. Extraer las probabilidades asignadas por tu modelo entrenado
    probabilidades = modelo_gb.predict_proba(X_partido[features])[0]
    
    p_empate = probabilidades[0]
    p_local = probabilidades[1]
    p_visitante = probabilidades[2]

    print(f"Probabilidad de Victoria de {local}: {p_local:.2f}%")
    print(f"Probabilidad de Empate: {p_empate:.2f}%")
    print(f"Probabilidad de Victoria de {visitante}: {p_visitante:.2f}%\n")
    
    return {'local': p_local, 'empate': p_empate, 'visitante': p_visitante}

# Escribe cualquier partido que aparezca en el fixture (Respeta las mayúsculas del CSV original)
_ = predecir_encuentro_mundial('Portugal', 'Spain')



Probabilidad de Victoria de Portugal: 0.22%
Probabilidad de Empate: 0.43%
Probabilidad de Victoria de Spain: 0.35%

